# Macroeconomic Determinants of US Housing Starts (1987–2024)

## Introduction

This project aims to examine the housing market model proposed by DiPasquale and Wheaton (1996).
Their model explains housing activity through demand factors, such as household income and employment stability, and supply factors, including construction costs and credit availability.
According to this model, demand decreases when economic problems, such as rising unemployment, weaken household purchasing power, leading to fewer buyers and lower housing activity. On the supply side, inflation raises mortgage rates and construction costs, making homes less affordable and limiting new construction.

My research question is: which macroeconomic factors explain the monthly variation in
US housing starts between 1987 and 2024?

The data is sourced from the Federal Reserve Economic Data (FRED), published by the
Federal Reserve Bank of St. Louis. It contains 445 monthly observations from January 1987 to January 2024. I use six macroeconomic variables as potential explanatory factors: 

Mortgage rate: higher rates make borrowing more expensive, reducing demand
Unemployment rate:  a weaker labour market means fewer households can afford to build
Real income: richer households demand more housing
Real house price: higher prices may incentivise more construction
Real construction cost: costlier inputs reduce the incentive to build

Where possible, variables are log-transformed (marked with l_ in the data).
Mortgage rate and unemployment are kept in their original percentage-point units
because logging rates is less meaningful economically.

## Imports and parameters

In [ ]:
# Import helper functions from utils.py and plotting library
import os, sys
from pathlib import Path

# Use the folder where this notebook is located
base = Path.cwd()
os.chdir(base)
sys.path.insert(0, str(base))

print("Working directory:", base)

In [ ]:
# Define file path, outcome variable and predictors
DATA_PATH  = "data/clean_model_data.csv"
OUTCOME    = "l_housing_starts"
PREDICTORS = ["l_real_price", "mortgage_rate", "unemployment", "l_income", "l_real_cost"]

## Data acquisition and processing

The data is read from a CSV file downloaded from FRED, with the date column parsed as a
monthly DatetimeIndex (1987-01 to 2024-01).

### Regression equation

$$
\log(\text{HousingStarts}_t) = \beta_0 + \beta_1 \log(\text{RealPrice}_t) + \beta_2 \text{MortgageRate}_t + \beta_3 \text{UnemploymentRate}_t + \beta_4 \log(\text{RealIncome}_t) + \beta_5 \log(\text{RealCost}_t) + \varepsilon_t
$$

### Variable transformations

This is a mixed log-level model, not all variables share the same unit:

- Log-transformed variables (`l_real_price`, `l_income`, `l_real_cost`, `l_housing_starts`):
  Taking logs means regression coefficients are elasticities, a 1% increase in X
  is associated with a β% change in housing starts.

- Level variables (`mortgage_rate`, `unemployment`): these are already in percentage
  points and logging rates is less economically meaningful, so they are kept in levels.
  Their coefficients are semi-elasticities, a 1 percentage-point rise in X is
  associated with a (β × 100)% change in housing starts.

This distinction is important when interpreting the results in the conclusion.

In [ ]:
# Load data and descriptive part 
df = load_and_prepare(DATA_PATH)
print(f"Shape: {df.shape}  |  {df.index.min().date()} to {df.index.max().date()}")
df.describe().round(3)

## Visualisation

The chart below shows log housing starts, mortgage rates and unemployment over time.
Two things stand out: the sharp collapse in housing starts during the 2008-09 financial
crisis coincides with rising unemployment, and a smaller dip appears during COVID-19 in 2020.
This suggests both variables are likely important predictors.

The scatter plots then show the raw relationship between each predictor and log housing starts,
giving us a sense of linearity before modelling.

In [ ]:
# Plot key variables over time to spot trends and structural breaks
fig = plot_time_series(df,
    columns=["l_housing_starts", "mortgage_rate", "unemployment"],
    title="Log Housing Starts, Mortgage Rate & Unemployment (1987-2024)")
plt.show()

In [ ]:
# Plot each predictor against log housing starts to check for linear relationships
fig2 = plot_scatter_matrix(df, predictors=PREDICTORS)
plt.show()


## Modelling

I run an OLS regression of log housing starts on the five predictors.
Standard errors are HC3 heteroscedasticity-robust, which is preferred over the default
homoscedastic standard errors for long macroeconomic time series where residual variance
tends to vary with the business cycle.

In [ ]:
# Run OLS regression and display full results table
model = run_ols(df, outcome=OUTCOME, predictors=PREDICTORS)
model.summary()

In [ ]:
# Check residuals to assess whether model assumptions hold
fig3 = plot_residuals(model.fittedvalues, model.resid)
plt.show()

## Conclusion

**Overall model fit:**

The F test p-value is essentially zero. The predictors are statistically significant. R² ≈ 0.70, meaning the model explains about 70% of the variance in log housing starts.

**Interpreting each coefficient:**

Mortgage rate (level): significant negative coefficient. A 1 percentage point rise
  in the mortgage rate is associated with a meaningful fall in housing starts which is consistent with theory. 

Unemployment (level): significant negative coefficient. A weaker labour market reduces
  housing demand, as households are less confident about future income.

Log real income (log): significant positive coefficient. A 1% rise in real income is
  associated with a β% rise in housing starts — richer households can afford more housing.

Log real construction cost (log): negative coefficient. Higher input costs reduce
  the incentive to build.

Log real house price (log): the sign here is worth discussing. Higher prices should
  in theory incentivise more building (supply response), but the estimate may be biased
  downward because of endogeneity (see weaknesses below).

**Weaknesses:**

Endogeneity: housing starts affect prices (more supply lowers prices), so OLS
  estimates for real price are likely biased. An instrumental-variables would be needed to isolate the causal effect.

Serial autocorrelation: Monthly time-series data tends to be autocorrelated.

Structural breaks: the 2008 crisis and the 2020 pandemic are large changes. Adding crisis dummies or a rolling-window model would test whether coefficients are
  stable over time.

## References

DiPasquale, D., & Wheaton, W. C. (1996). Urban Economics and Real Estate Markets. Prentice Hall.

Federal Reserve Bank of St. Louis. (2024). Federal Reserve Economic Data (FRED). Retrieved from https://fred.stlouisfed.org